In [ ]:
# ============================================================
# notebooks/08_fusion_model_training.py
# Run after 07_contextual_features.py:
#   python notebooks/08_fusion_model_training.py
#
# PURPOSE: Two-stage notebook:
#   Stage 1 — Generate real Phase 1 risk scores for all 590K rows
#             using the full EARN+ pipeline (AE + ResNet → 128 →
#             Nystroem → IPCA → AttentionRXTJ + IFM ensemble).
#             Patches contextual_features.npy column 7 in-place.
#   Stage 2 — Train FusionNet v2 (8→64→32→16→1 with feature attention)
#             using Jaya threshold optimisation. Saves fusion_net.pt,
#             fusion_scaler.pkl, fusion_config.json, ROC plots.
#
# WHAT WAS FIXED (merged from fix_retrain_nb08.py):
#   - AE architecture: added Dropout at indices [3] and [7] and removed
#     the spurious extra ReLU after the final encoder Linear — these exact
#     positions match the autoencoder.pt state_dict keys (encoder.0,1,4,5,8)
#   - Pipeline order: raw (224) → AE(64) + ResNet(64) → concat(128) →
#     Nystroem(128→300) → IPCA(300→50) — NOT raw(224) → Nystroem directly
#   - AttentionRXTJ: uses exact app.py names (paths ModuleList, attn_gru,
#     attention Sequential, classifier Sequential) so load_state_dict works
#   - FusionNet v2: wider (8→64→32→16→1) + CosineAnnealingLR (removes the
#     deprecated verbose=True ReduceLROnPlateau) + monitors AUC not loss
#   - Training: EPOCHS=200, PATIENCE=25, batch=4096, lr=5e-4
#   - Smart resume: skips Stage 1 if model_probs_full.npy already exists
#     with the correct row count
#
# OUTPUTS:
#   models/fusion_net.pt          — trained FusionNet v2 weights
#   models/fusion_scaler.pkl      — StandardScaler for 8 input features
#   results/fusion_config.json    — thresholds + metrics + attn weights
#   results/fusion_roc.png        — ROC curve + attention weight chart
#   results/fusion_training_curves.png
#   data/model_probs_full.npy     — P1 risk scores for all 590K rows
#
# RUNTIME: ~25 min (Stage 1) + ~15 min (Stage 2) = ~40 min total
#          Stage 1 skipped if model_probs_full.npy already exists.
# ============================================================

In [ ]:
# %% Requirements Cell: Establish backend and ensure standard scikit-learn footprint
%pip install pandas pyarrow numpy joblib torch scikit-learn matplotlib

In [ ]:
# %% Cell 1: Imports
import os
import sys
import json
import time
import warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use("Agg")  # Set non-interactive backend to eliminate display runtime warnings
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, matthews_corrcoef,
                             precision_score, recall_score, f1_score,
                             roc_curve, confusion_matrix)

ROOT        = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR    = os.path.join(ROOT, "data")
MODEL_DIR   = os.path.join(ROOT, "models")
RESULTS_DIR = os.path.join(ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
DEVICE = torch.device("cpu")

print(f"[NB08] Path Context Assigned: {ROOT}")
print(f"[NB08] Engine Target Runtime: {DEVICE}")

# ════════════════════════════════════════════════════════════════════════════════
# CORE ARCHITECTURE BLUEPRINTS (Mirrors deployment environment layouts)
# ════════════════════════════════════════════════════════════════════════════════

class FraudAutoencoder(nn.Module):
    def __init__(self, input_dim=224, latent_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 128),       nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128, 256),        nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, input_dim)
        )
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z
    def encode(self, x):
        return self.encoder(x)

class _ResBlock(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(in_dim,  out_dim), nn.BatchNorm1d(out_dim), nn.ReLU(),
            nn.Linear(out_dim, out_dim), nn.BatchNorm1d(out_dim)
        )
        self.shortcut = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()
        self.relu     = nn.ReLU()
    def forward(self, x):
        return self.relu(self.block(x) + self.shortcut(x))

class FraudResNet(nn.Module):
    def __init__(self, input_dim=224):
        super().__init__()
        self.stem   = nn.Linear(input_dim, 128)
        self.blocks = nn.Sequential(
            _ResBlock(128, 128),
            _ResBlock(128, 64),
            _ResBlock(64,  64),
            _ResBlock(64,  64),
        )
        self.head = nn.Linear(64, 2)
    def extract(self, x):
        return self.blocks(torch.relu(self.stem(x)))
    def forward(self, x):
        return self.head(self.extract(x))

SEQ_LEN     = 8
CARDINALITY = 4

class _ResNeXtBlock(nn.Module):
    def __init__(self, in_dim, out_dim, cardinality=CARDINALITY):
        super().__init__()
        gd = out_dim // cardinality
        self.paths    = nn.ModuleList([
            nn.Sequential(
                nn.Linear(in_dim, gd), nn.BatchNorm1d(gd), nn.ReLU(),
                nn.Linear(gd, gd),     nn.BatchNorm1d(gd)
            ) for _ in range(cardinality)
        ])
        self.shortcut = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()
        self.relu     = nn.ReLU()
    def forward(self, x):
        return self.relu(torch.cat([p(x) for p in self.paths], dim=-1) + self.shortcut(x))

class _ResNeXtExtractor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            _ResNeXtBlock(input_dim, 128), _ResNeXtBlock(128, 128),
            _ResNeXtBlock(128, 64),        _ResNeXtBlock(64,  64)
        )
    def forward(self, x):
        return self.net(x)

class _SelfAttentionGRU(nn.Module):
    def __init__(self, input_dim=64, hidden_dim=64, seq_len=SEQ_LEN):
        super().__init__()
        self.seq_len  = seq_len
        self.step_dim = input_dim // seq_len
        self.gru      = nn.GRU(self.step_dim, hidden_dim, num_layers=2, batch_first=True, dropout=0.3)
        self.attention  = nn.Sequential(nn.Linear(hidden_dim, 32), nn.Tanh(), nn.Linear(32, 1))
        self.classifier = nn.Sequential(nn.Linear(hidden_dim, 32), nn.ReLU(), nn.Dropout(0.3), nn.Linear(32, 1), nn.Identity())
    def forward(self, x):
        x       = x.view(x.size(0), self.seq_len, self.step_dim)
        out, _  = self.gru(x)
        attn_w  = torch.softmax(self.attention(out), dim=1)
        context = (attn_w * out).sum(dim=1)
        return torch.sigmoid(self.classifier(context)).squeeze(1), attn_w

class AttentionRXTJ(nn.Module):
    def __init__(self, input_dim, seq_len=SEQ_LEN):
        super().__init__()
        self.resnext  = _ResNeXtExtractor(input_dim)
        self.attn_gru = _SelfSelfAttentionGRU = _SelfAttentionGRU(input_dim=64, seq_len=seq_len)
    def forward(self, x):
        return self.attn_gru(self.resnext(x))

class FusionNet(nn.Module):
    def __init__(self, input_dim=8):
        super().__init__()
        self.feature_attn = nn.Parameter(torch.ones(input_dim))
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32),        nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 16),        nn.ReLU(),
            nn.Linear(16, 1)
        )
    def forward(self, x):
        w = torch.softmax(self.feature_attn, dim=0)
        return torch.sigmoid(self.net(x * w)).squeeze(1), w

def _logits(m, x):
    return m.net(x * torch.softmax(m.feature_attn, dim=0)).squeeze(1)

In [ ]:
# %% Cell 2: Stage 1 Verification & Scoring Execution
P1_PROBS_PATH = os.path.join(DATA_DIR, "model_probs_full.npy")
SNAP_PATH     = os.path.join(DATA_DIR, "tx_snapshot.parquet")
CTX_PATH      = os.path.join(DATA_DIR, "contextual_features.npy")

_need_p1 = True
if os.path.exists(P1_PROBS_PATH):
    _existing = np.load(P1_PROBS_PATH)
    _snap_len = len(pd.read_parquet(SNAP_PATH, columns=["TransactionID"]))
    if len(_existing) == _snap_len and np.std(_existing) > 0.01:
        print(f"\n[STAGE 1] Smart Resume Active — Utilizing existing model_probs_full.npy ({len(_existing):,} rows)")
        p1_scores = _existing.astype(np.float32)
        _need_p1  = False

if _need_p1:
    print("\n" + "="*60)
    print("  RUNNING STAGE 1: Generating Comprehensive Phase 1 Risk Scores")
    print("="*60)
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        imputer  = joblib.load(os.path.join(MODEL_DIR, "imputer.pkl"))
        scaler   = joblib.load(os.path.join(MODEL_DIR, "scaler.pkl"))
        nystroem = joblib.load(os.path.join(MODEL_DIR, "nystroem.pkl"))
        ipca     = joblib.load(os.path.join(MODEL_DIR, "incremental_pca.pkl"))
        ifm      = joblib.load(os.path.join(MODEL_DIR, "isolation_forest.pkl"))

    # Runtime check for scikit-learn version disparities
    if not hasattr(imputer, "_fill_dtype"):
        imputer._fill_dtype = np.float32

    RAW_DIM  = int(imputer.n_features_in_)
    NYS_DIM  = int(nystroem.n_features_in_)
    IPCA_DIM = int(ipca.n_components_)

    ae_state = torch.load(os.path.join(MODEL_DIR, "autoencoder.pt"), map_location="cpu")
    ae_model = FraudAutoencoder(input_dim=ae_state["encoder.0.weight"].shape[1], latent_dim=64)
    ae_model.load_state_dict(ae_state)
    ae_model.eval()

    rn_model = FraudResNet(input_dim=ae_state["encoder.0.weight"].shape[1])
    rn_state = torch.load(os.path.join(MODEL_DIR, "resnet_extractor.pt"), map_location="cpu")
    rn_model.load_state_dict(rn_state, strict=False)
    rn_model.eval()

    p1_model = AttentionRXTJ(input_dim=IPCA_DIM, seq_len=SEQ_LEN).to(DEVICE)
    p1_state = torch.load(os.path.join(MODEL_DIR, "attention_rxtj.pt"), map_location=DEVICE)
    p1_model.load_state_dict(p1_state)
    p1_model.eval()

    cfg_p1    = json.load(open(os.path.join(RESULTS_DIR, "deployment_config.json")))
    W_MODEL   = float(cfg_p1["W_MODEL"])
    W_IFM     = float(cfg_p1["W_IFM"])
    THRESHOLD = float(cfg_p1["THRESHOLD"])

    snap    = pd.read_parquet(SNAP_PATH)
    pcd_map = {"W": 0, "H": 1, "C": 2, "S": 3, "R": 4}
    X_raw   = np.full((len(snap), RAW_DIM), np.nan, dtype=np.float32)
    X_raw[:, 0] = snap["TransactionAmt"].fillna(0).values.astype(np.float32)
    X_raw[:, 1] = snap["hour"].values.astype(np.float32)
    X_raw[:, 2] = snap["ProductCD"].map(pcd_map).fillna(-1).values.astype(np.float32)
    X_raw[:, 3] = pd.to_numeric(snap["addr1"], errors="coerce").values.astype(np.float32)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        X_sc = scaler.transform(imputer.transform(X_raw)).astype(np.float32)

    BATCH = 4096
    Z_ae_list, Z_rn_list = [], []
    with torch.no_grad():
        for s in range(0, len(X_sc), BATCH):
            xb = torch.FloatTensor(X_sc[s:s+BATCH])
            Z_ae_list.append(ae_model.encode(xb).cpu().numpy())
            Z_rn_list.append(rn_model.extract(xb).cpu().numpy())

    Z_earn = np.hstack([np.vstack(Z_ae_list), np.vstack(Z_rn_list)]).astype(np.float32)
    
    CHUNK  = 8000
    X_ipca = np.zeros((len(Z_earn), IPCA_DIM), dtype=np.float32)
    for s in range(0, len(Z_earn), CHUNK):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            X_ipca[s:s+CHUNK] = ipca.transform(nystroem.transform(Z_earn[s:s+CHUNK])).astype(np.float32)

    all_risk = []
    with torch.no_grad():
        for s in range(0, len(X_ipca), BATCH):
            probs, _ = p1_model(torch.FloatTensor(X_ipca[s:s+BATCH]).to(DEVICE))
            ifm_norm = 1.0 / (1.0 + np.exp(ifm.decision_function(X_ipca[s:s+BATCH])))
            all_risk.extend((W_MODEL * probs.cpu().numpy() + W_IFM * ifm_norm).tolist())

    p1_scores = np.array(all_risk, dtype=np.float32)
    np.save(P1_PROBS_PATH, p1_scores)

X_ctx       = np.load(CTX_PATH)
X_ctx[:, 7] = p1_scores
np.save(CTX_PATH, X_ctx)
print(f"[STAGE 1 Complete] In-place replacement successful. Target Vector Boundaries: {X_ctx[:, 7].min():.4f}–{X_ctx[:, 7].max():.4f}")

In [ ]:
# %% Cell 3: Ingest Data Elements
print("\n" + "="*60)
print("  STAGE 2: Initializing Deep Fusion Target Pipelines")
print("="*60)

X        = np.load(CTX_PATH)
y        = np.load(os.path.join(DATA_DIR, "fusion_labels.npy"))
acct_ids = np.load(os.path.join(DATA_DIR, "fusion_account_ids.npy"))
with open(os.path.join(DATA_DIR, "fusion_feature_names.json")) as f:
    FEATURE_NAMES = json.load(f)

print(f"  Matrix footprint loaded: X={X.shape} Labels={int(y.sum()):,} matches.")
X = np.nan_to_num(X, nan=0.0)

In [ ]:
# %% Cell 4: Scaling Vectors
fusion_scaler = StandardScaler()
X_sc          = fusion_scaler.fit_transform(X).astype(np.float32)
joblib.dump(fusion_scaler, os.path.join(MODEL_DIR, "fusion_scaler.pkl"))
print("  ✓ Unified tracking scalars saved: models/fusion_scaler.pkl")

In [ ]:
# %% Cell 5: Stratified Data Splitting Process
unique_a = np.unique(acct_ids)
fa       = set(acct_ids[y == 1])
al       = np.array([1 if a in fa else 0 for a in unique_a])

a_tr, a_tmp, _, _ = train_test_split(unique_a, al, test_size=0.30, stratify=al, random_state=42)
a_va, a_te, _, _  = train_test_split(a_tmp, np.array([1 if a in fa else 0 for a in a_tmp]), test_size=0.50, random_state=42)

tr_m = np.isin(acct_ids, a_tr)
va_m = np.isin(acct_ids, a_va)
te_m = np.isin(acct_ids, a_te)

X_tr, y_tr = X_sc[tr_m], y[tr_m]
X_va, y_va = X_sc[va_m], y[va_m]
X_te, y_te = X_sc[te_m], y[te_m]

print(f"  Partition matrices: Train={len(X_tr):,} | Val={len(X_va):,} | Test={len(X_te):,}")

In [ ]:
# %% Cell 6: Network Structural Formulation Setup
INPUT_DIM = X_tr.shape[1]
model_f   = FusionNet(INPUT_DIM).to(DEVICE)

pos_weight = torch.tensor([(y_tr==0).sum() / (y_tr==1).sum()]).float()
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))
optimizer  = torch.optim.Adam(model_f.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200, eta_min=1e-5)

BATCH_SIZE = 4096
EPOCHS     = 200
PATIENCE   = 25

loader = DataLoader(TensorDataset(torch.FloatTensor(X_tr), torch.FloatTensor(y_tr)), batch_size=BATCH_SIZE, shuffle=True)
Xv_t   = torch.FloatTensor(X_va).to(DEVICE)
yv     = y_va

In [ ]:
# %% Cell 7: Network Tuning Optimization Steps
print(f"[Optimization Engine Execution] Target Metrics Objective: Validation AUC ≥ 0.85")
best_auc, best_state, no_imp = 0.0, None, 0
train_losses, val_aucs = [], []

for ep in range(1, EPOCHS + 1):
    model_f.train()
    ep_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(_logits(model_f, xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model_f.parameters(), 1.0)
        optimizer.step()
        ep_loss += loss.item() * len(xb)
    scheduler.step()

    model_f.eval()
    with torch.no_grad():
        vp = torch.sigmoid(_logits(model_f, Xv_t)).cpu().numpy()
    vauc = roc_auc_score(yv, vp)

    train_losses.append(ep_loss / len(X_tr))
    val_aucs.append(vauc)

    if ep % 10 == 0 or ep <= 5:
        print(f"  Step Vector {ep:>3}/{EPOCHS} | Train Loss={ep_loss/len(X_tr):.4f} | Val AUC={vauc:.4f}")

    if vauc > best_auc:
        best_auc = vauc
        no_imp = 0
        best_state = {k: v.clone() for k, v in model_f.state_dict().items()}
    else:
        no_imp += 1
        if no_imp >= PATIENCE:
            print(f"  Early stopping criteria satisfied at epoch {ep}.")
            break

model_f.load_state_dict(best_state)
model_f.eval()
print(f"  ✓ Model parameter state localized. Stable Peak Target Validation AUC: {best_auc:.4f}")

In [ ]:
# %% Cell 8: Jaya Threshold Matrix Search
with torch.no_grad():
    vp_np = torch.sigmoid(_logits(model_f, Xv_t)).cpu().numpy()

def _cost(t, p, l):
    pred = (p >= t).astype(int)
    fp   = ((pred==1) & (l==0)).sum()
    fn   = ((pred==0) & (l==1)).sum()
    tp   = ((pred==1) & (l==1)).sum()
    tn   = ((pred==0) & (l==0)).sum()
    return 2.0*fp/(fp+tn+1e-9) + fn/(fn+tp+1e-9)

pop = np.random.uniform(0.1, 0.9, 40)
c   = np.array([_cost(t, vp_np, yv) for t in pop])
for _ in range(150):
    bi, wi = np.argmin(c), np.argmax(c)
    r1, r2 = np.random.rand(40), np.random.rand(40)
    np2    = np.clip(pop + r1*(pop[bi]-np.abs(pop)) - r2*(pop[wi]-np.abs(pop)), 0.05, 0.95)
    nc     = np.array([_cost(t, vp_np, yv) for t in np2])
    m      = nc < c; pop = np.where(m, np2, pop); c = np.where(m, nc, c)

OPT_T  = float(pop[np.argmin(c)])
HIGH_T = min(OPT_T + 0.15, 0.90)
ELEV_T = max(OPT_T - 0.10, 0.25)

print(f"  [Calculated Parameters] Operational Base Threshold   : {OPT_T:.4f}")
print(f"  [Calculated Parameters] Elevated Category Threshold : {ELEV_T:.4f}")
print(f"  [Calculated Parameters] Critical Range High Boundary : {HIGH_T:.4f}")

In [ ]:
# %% Cell 9: Evaluation Pipeline
Xt_t = torch.FloatTensor(X_te).to(DEVICE)
with torch.no_grad():
    test_probs = torch.sigmoid(_logits(model_f, Xt_t)).cpu().numpy()
    attn_final = torch.softmax(model_f.feature_attn, dim=0).cpu().numpy()

test_preds = (test_probs >= OPT_T).astype(int)
auc  = roc_auc_score(y_te, test_probs)
mcc  = matthews_corrcoef(y_te, test_preds)
prec = precision_score(y_te, test_preds, zero_division=0)
rec  = recall_score(y_te, test_preds, zero_division=0)
f1   = f1_score(y_te, test_preds, zero_division=0)
cm   = confusion_matrix(y_te, test_preds)

TP, FP, FN, TN = int(cm[1,1]), int(cm[0,1]), int(cm[1,0]), int(cm[0,0])
print(f"\n  ┌─────────────────────────────────────────┐")
print(f"  │  AUC-ROC Verification Score : {auc:.4f}      │")
print(f"  │  MCC Evaluation Metric      : {mcc:.4f}      │")
print(f"  │  Precision Value Matrix     : {prec:.4f}      │")
print(f"  │  Recall Target Extraction   : {rec:.4f}      │")
print(f"  │  Balanced F1 Context Score  : {f1:.4f}      │")
print(f"  │  Outcomes: TP={TP:<5} FP={FP:<5} TN={TN:<5} FN={FN:<5} │")
print(f"  └─────────────────────────────────────────┘")

In [ ]:
# %% Cell 10: Performance Renderings
fpr_a, tpr_a, _ = roc_curve(y_te, test_probs)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(fpr_a, tpr_a, color="#0099cc", lw=2, label=f"FusionNet v2 AUC={auc:.4f}")
axes[0].plot([0,1],[0,1],"k--",lw=0.8)
axes[0].axvline(x=0.15, color="gray", ls=":", alpha=0.5, label="15% FPR Target Anchor")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("FusionNet v2 ROC Curve Line Evaluation")
axes[0].legend()

axes[1].bar(FEATURE_NAMES, attn_final, color="#00d4ff", edgecolor="#0099cc")
axes[1].set_title("Learned Multi-Feature Attention Distribution Weights")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fusion_roc.png"), dpi=150, bbox_inches="tight")
plt.close()

fig2, ax2 = plt.subplots(figsize=(8, 4))
ax2.plot(train_losses, label="Tracking Training Loss Bounds", color="#0099cc")
ax2.plot(val_aucs,     label="Tracking Validation Area Curves (AUC)",    color="#00e5a0")
ax2.set_xlabel("Operational Epochs Step Index")
ax2.set_title("FusionNet v2 Converging Profile Paths")
ax2.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fusion_training_curves.png"), dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ Analytics diagrams rendered cleanly inside target results paths.")

In [ ]:
# %% Cell 11: Final Asset Serialization
torch.save(model_f.state_dict(), os.path.join(MODEL_DIR, "fusion_net.pt"))

fusion_config = {
    "input_dim":            INPUT_DIM,
    "fusion_threshold":     round(OPT_T, 6),
    "high_threshold":       round(HIGH_T, 6),
    "elevated_threshold":   round(ELEV_T, 6),
    "fusion_auc":           round(float(auc), 6),
    "fusion_mcc":           round(float(mcc), 6),
    "fusion_precision":     round(float(prec), 6),
    "fusion_recall":        round(float(rec), 6),
    "fusion_f1":            round(float(f1), 6),
    "true_positives":       TP,
    "false_positives":      FP,
    "true_negatives":       TN,
    "false_negatives":      FN,
    "feature_names":        FEATURE_NAMES,
    "optimal_attn_weights": [round(float(w), 6) for w in attn_final],
    "model_version":        "fusionnet_v2",
    "training_epochs":      len(train_losses),
    "best_val_auc":         round(float(best_auc), 6),
    "pos_weight_used":      round(float(pos_weight.item()), 4),
}

with open(os.path.join(RESULTS_DIR, "fusion_config.json"), "w") as f:
    json.dump(fusion_config, f, indent=2)

print("\n[NB08 Execution Run Complete]")
print(f"  Models Saved  → models/fusion_net.pt")
print(f"  Config Saved  → results/fusion_config.json")
print(f"  Next step validation sequence path → integrate app_phase2_endpoints.py endpoints into app.py layer.")